# Data Cleaning Pipeline
Takes in a run name (see notebook 0), cleans off the spinup time and removes values that we do not want the neural network to see. Aggregates into an array. Segmented so the next notebook doesn't change much. Takes a while to run (although much fast with scipy!)

In [1]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import netCDF4
from pathlib import Path
from datetime import datetime
import json
import re
import time

In [2]:
import scipy

In [3]:
from helpers.data_funcs import read_mod_file #need to restart kernel to get these updated!!
from helpers.data_funcs import check_nans

In [4]:
## Config block, change things here
RUN_NAME = "ksno16_ktherm"
CLEAN_NAME = "ksno16_ktherm"

forcing_loc_list = ["centralarctic", "coastalcanada", "sibchuk", "barents"]
saved_vars = ['evap','fsnow','frazil','fswabs','flw','flwout','fsens','fsurf','flat','frain','Tair','Qa','fsw','fcondtop','meltt','meltb','meltl','snoice','dsnow','congel','sst','sss','Tf','fhocn','melts', 'aicen','vicen', 'vsnon']
new_vars = ["Tsnon10", "Tsnon11", "Tsnon12"]  
saved_tracers_dict = {"Tsfcn":[1],"qice":[2,3,4,5,6,7,8,9],"qsno": [10, 11, 12], "sice":[13,14,15,16,17,18,19,20]} #using fortran/.sel indexing, check log file for tracer defs
changed_param = ["ksno"]
changed_parameterizations = ["ktherm"]
start_year = 1975
spinup_years=0
agg_hours = 24

In [5]:
rhos = 330 # kg/m^3
cp = 2106 # J/kg/c
Lf = 2.835e6 - 2.501e6 # J/kg
SCRATCH_DIR = Path(f"/scratch/alpine/dagr5998/{RUN_NAME}")
PARAM_DIR = Path(f"/projects/dagr5998/icepack_ML/parameters/{RUN_NAME}")  
OUT_DIR = Path(f"/projects/dagr5998/cleaned_data/01icepack_array_output/{CLEAN_NAME}")

In [6]:
def snow_temp(q, vsnon, tsfcn):
    # snow layer temp, if enthalpy is 0 then we use surface temp
    T = (q + Lf * rhos) / (cp * rhos)
    snow_mask = (vsnon > 0) & (q < 0) # if q = 0 then enthalpy isn't set
    return T.where(snow_mask, tsfcn)

In [7]:
def clean_run(path, saved_vars, saved_tracers, spinup_years=2):
    # Takes the path to the dataset and basic inputs and retearns cleaned dataset
    #with xr.open_dataset(path, decode_times=False) as ds:
    #    ds = ds[saved_vars + ["trcrn"]].load()
    ds = xr.open_dataset(path, engine="scipy", decode_times=False)
    ds = ds.sel(ni=3).drop_vars("ni").astype("float32")

    out = {v: ds[v] for v in saved_vars}
    for name, idxs in saved_tracers.items():
        for i in idxs:
            out[f"{name}{i}"] = ds["trcrn"].sel(ntrcr=i).drop_vars("ntrcr")
    for i in saved_tracers["qsno"]:                               # 10, 11, 12
            out[f"Tsnon{i}"] = snow_temp(out[f"qsno{i}"], out["vsnon"], out["Tsfcn1"])
            
    return xr.Dataset(out).isel(time=slice(spinup_years * 365 * 24, None))

In [8]:
def aggregate(ds, hours=24):
    #cuts down on memory, keeps trend
    if hours == 1:
        return ds
    return ds.coarsen(time=hours, boundary="trim").mean()

In [9]:
def to_array(ds, saved_vars, saved_tracers_dict, new_vars=[]):
    #Takes in a cleaned xarray netcdf dataset and the pulled variables and puts it into (N, T, C): (Number of runs, Timesteps, Channel (variables)) 
    names = list(saved_vars) + [f"{n}{i}" for n, index in saved_tracers_dict.items() for i in index] + list(new_vars)
    #print(names)
    ncat = ds.sizes["ncat"]
    cols, channels = [], []
    for i in names:
        ds_i = ds[i]
        if "ncat" in ds_i.dims:
            cols.append(ds_i.transpose("time", "ncat").values)
            channels += [f"{i}_ncat{c}" for c in ds_i["ncat"].values]
        else:
            cols.append(ds_i.values[:,None])
            channels += [i]
    return np.concatenate(cols, axis=1).astype(np.float32), channels

In [10]:
run_dirs = sorted(SCRATCH_DIR.glob("*_set_*"))
#run_dirs = sorted(SCRATCH_DIR.glob("barents_set_*"))
parameterizations=[]
X_list, Y_list, sets, locations = [], [], [], []
channels = None
t0 = time.time()
count = 0
for run_dir in run_dirs:
    loc, set_name = run_dir.name.split("_", 1)
    params = read_mod_file(PARAM_DIR / f"{set_name}.mods")

    ds = next((run_dir / "history").glob("*.nc"))
    ds_i = aggregate(clean_run(ds, saved_vars, saved_tracers_dict,  spinup_years), agg_hours)
    X_i, channels_i = to_array(ds_i, saved_vars,saved_tracers_dict,new_vars)   # one run

    if channels is None:
        channels = channels_i
    assert channels_i == channels 

    X_list.append(X_i)
    Y_list.append([params[k] for k in changed_param])     
    sets.append(int(set_name.rsplit("_", 1)[1]))
    locations.append(loc)
    parameterizations.append([params[k] for k in changed_parameterizations])
    count +=1 
    print(f"{time.time() - t0:.0f}s elapsed, {count} run(s) done")
print(set([i.shape for i in X_list]))
X = np.stack(X_list) #(N, T, C) time series for every run
Y = np.array(Y_list, dtype=np.float32) #(N, P) parameter values for each run
sets = np.array(sets, dtype=np.int32)  #(N,) this is param/parameterization it was run on
locations = np.array(locations) #(N,) forcing loc

print(X.shape, Y.shape, sets.shape, locations.shape)

42s elapsed, 1 run(s) done
65s elapsed, 2 run(s) done
102s elapsed, 3 run(s) done
129s elapsed, 4 run(s) done
162s elapsed, 5 run(s) done
191s elapsed, 6 run(s) done
230s elapsed, 7 run(s) done
273s elapsed, 8 run(s) done
315s elapsed, 9 run(s) done
368s elapsed, 10 run(s) done
422s elapsed, 11 run(s) done
474s elapsed, 12 run(s) done
509s elapsed, 13 run(s) done
572s elapsed, 14 run(s) done
629s elapsed, 15 run(s) done
674s elapsed, 16 run(s) done
730s elapsed, 17 run(s) done
801s elapsed, 18 run(s) done
853s elapsed, 19 run(s) done
927s elapsed, 20 run(s) done
994s elapsed, 21 run(s) done
1062s elapsed, 22 run(s) done
1202s elapsed, 23 run(s) done
1337s elapsed, 24 run(s) done
1392s elapsed, 25 run(s) done
1445s elapsed, 26 run(s) done
1486s elapsed, 27 run(s) done
1535s elapsed, 28 run(s) done
1659s elapsed, 29 run(s) done
1708s elapsed, 30 run(s) done
1813s elapsed, 31 run(s) done
1930s elapsed, 32 run(s) done
1987s elapsed, 33 run(s) done
2030s elapsed, 34 run(s) done
2075s elapse

{(17885, 155)}


In [12]:
OUT_DIR.mkdir(parents=True)

FileExistsError: [Errno 17] File exists: '/projects/dagr5998/cleaned_data/01icepack_array_output/ksno16_ktherm'

In [13]:
np.save(OUT_DIR / "X.npy", X)
np.save(OUT_DIR/ "Y.npy", Y)
np.save(OUT_DIR / "sets.npy", sets)
np.save(OUT_DIR/ "locations.npy", locations)
np.save(OUT_DIR / "parameterizations.npy", np.array(parameterizations))
config = {
    "channels":     channels,
    "parameters": changed_param,
    "parameterizations":  changed_parameterizations,
    "shape":        list(X.shape),
    "start_year":         start_year,
    "first_year":         start_year + spinup_years,
    "source_dir":   str(SCRATCH_DIR),
    "saved_vars":    saved_vars,
    "new_vars": new_vars,
    "saved_tracers": saved_tracers_dict,
    "aggregated hours": agg_hours,
    "created":       datetime.now().isoformat(timespec="seconds"),
    "notes": {
        "tracer_naming": "Tracer channel numbers are Fortran ntrcr indices from the run log, not layer numbers. qice2-qice9 are ice layers 1-8 are so forth ",
    },
}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=2)) #creates a record to go along with the output


4078